# T63 — True polar wander decomposition

**Cluster G: Paleomagnetism.**

Total plate motion measured relative to the deep mantle can be decomposed into two contributions: (1) individual plate motions relative to the underlying mantle, and (2) coherent rotation of the whole solid Earth (crust + mantle) relative to the spin axis — **true polar wander (TPW)**. TPW is what happens when mass redistribution inside the mantle changes the moment of inertia enough to rotate the whole planet's rotation axis with respect to its surface. It's small on human timescales but cumulatively tens of degrees across the Phanerozoic.

**Where this notebook sits in the paper's story.** T31 (paleo-mantle reference-frame uncertainty) shows that Z22's mantle frame and paleomag frame put continents in *different* positions at the same age. T61 (Leonard 2025) shows the downstream consequence: reference-frame choice shifts reconstructed climate by ~5-15 °C. **This notebook explains *why*** — the difference between the two frames IS the accumulated TPW rotation. If TPW were zero, the mantle and paleomag frames would be identical.

## What this notebook produces

1. **§3 — TPW rotation extraction.** At each age from 0 to `MAX_AGE_MA` in `STEP_MA` increments, compute Africa's finite rotation in Z22's mantle frame (anchor 0) vs its paleomag frame (anchor 701701). The difference IS the TPW rotation applied at that age.
2. **§4 — TPW pole path on a globe.** Plot the accumulated TPW Euler poles as a track on an orthographic paleo-Earth. Compares with the Steinberger & Torsvik (2008) published TPW curve.
3. **§5 — TPW rate through time.** Angular rate (° per Myr) of the incremental TPW rotation between adjacent time steps, plotted as a bar chart. Highlights the ~110 Ma "monster" TPW event and the mid-Cretaceous quiescence.

## Learning objectives

- Combine `pygplates.RotationModel` calls with different `anchor_plate_id` values to extract the accumulated TPW rotation.
- Decompose a `pygplates.FiniteRotation` into its (pole latitude, pole longitude, angle) representation.
- Compute the *incremental* rotation between two ages by composing the two accumulated rotations.
- Cross-check a computed TPW curve against a published (Steinberger & Torsvik 2008) benchmark.

## Prerequisites and runtime

- No bundled data — uses only the Z22 rotation file via `plate_model_manager`. Steinberger & Torsvik 2008 TPW curve is hard-coded from their Table 1 for visual comparison.
- Python: `gplately`, `pygplates`, `pygmt`, `pandas`, `numpy`, `matplotlib`.
- Runtime: ~5 s (rotation arithmetic is fast; only two pyGMT globes).


## Environment + imports


In [1]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd
import pygmt
import gplately
import pygplates
from plate_model_manager import PlateModelManager

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, pygmt, gplately, pygplates):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


In [2]:
# === USER CONFIGURATION =====================================================
MODEL_NAME               = "Zahirovic2022"

# The plate whose two-frame difference reveals TPW. Africa is the canonical
# choice — it's the plate onto which paleomagnetic constraints and mantle-frame
# constraints are both directly wired via 701 (Africa) → 701701 (Africa paleomag
# synthetic anchor in Z22). Any plate ID whose rotation differs between the two
# frames will yield the same TPW answer; Africa just has the cleanest chain.
PLATE_ID_MANTLE_FRAME    = 701          # Africa in Z22 default (mantle) frame
PLATE_ID_PALEOMAG_FRAME  = 701          # Africa in the paleomag-anchor chain
ANCHOR_MANTLE_FRAME      = 0            # Z22 default mantle anchor
ANCHOR_PALEOMAG_FRAME    = 701701       # Z22 synthetic paleomag anchor for Africa

# Time discretisation
MAX_AGE_MA               = 400          # cap at Z22's ~410 Ma coverage
STEP_MA                  = 5            # 5-Myr cadence
AGES                     = np.arange(0, MAX_AGE_MA + STEP_MA, STEP_MA)

# TPW-path map settings
POLE_PATH_REGION         = [-180, 180, -90, 90]
POLE_PATH_PROJ           = "G0/60/15c"   # orthographic centred at 60° N over Greenwich

# Rate-plot settings
RATE_BAR_WIDTH_MA        = STEP_MA * 0.8
# ============================================================================
print(f"  model:        {MODEL_NAME}")
print(f"  ages:         0-{MAX_AGE_MA} Ma at {STEP_MA}-Myr cadence ({len(AGES)} steps)")
print(f"  plate:        {PLATE_ID_MANTLE_FRAME} (Africa)")
print(f"  mantle anchor: {ANCHOR_MANTLE_FRAME}")
print(f"  paleomag anchor: {ANCHOR_PALEOMAG_FRAME}")


  model:        Zahirovic2022
  ages:         0-400 Ma at 5-Myr cadence (81 steps)
  plate:        701 (Africa)
  mantle anchor: 0
  paleomag anchor: 701701


## 1. Load the Z22 rotation model


In [3]:
pmm      = PlateModelManager()
model    = pmm.get_model(MODEL_NAME, data_dir="data/pmm_cache")
rot_files = model.get_rotation_model()   # list of .rot filepaths or a RotationModel

# pygplates.RotationModel accepts a list of file paths OR a RotationModel instance
rot_model = pygplates.RotationModel(rot_files)
print(f"  Loaded {MODEL_NAME} rotation model")


  Loaded Zahirovic2022 rotation model


## 2. Extract the TPW rotation at each age

At age *t*, we ask pygplates for two finite rotations:

- **R_mantle(t)** = rotation from present-day Africa to Africa-at-time-*t* in the mantle frame.
- **R_paleomag(t)** = rotation from present-day Africa to Africa-at-time-*t* in the paleomag frame.

Both start from the same present-day Africa, but end up at different absolute positions because the mantle frame differs from the paleomag frame by cumulative TPW. So:

**R_TPW(t)** = R_mantle(t) × R_paleomag(t)⁻¹

is the rotation that, applied to a paleomag-frame position, gives you the mantle-frame position at that age. That IS the accumulated TPW correction from 0 to *t*.


In [4]:
def get_accumulated_tpw(age_ma):
    """Return the accumulated TPW FiniteRotation from 0 to age_ma."""
    r_mantle = rot_model.get_rotation(
        to_time=float(age_ma),
        moving_plate_id=PLATE_ID_MANTLE_FRAME,
        anchor_plate_id=ANCHOR_MANTLE_FRAME,
    )
    r_paleomag = rot_model.get_rotation(
        to_time=float(age_ma),
        moving_plate_id=PLATE_ID_PALEOMAG_FRAME,
        anchor_plate_id=ANCHOR_PALEOMAG_FRAME,
    )
    # TPW = mantle_frame_position × (paleomag_frame_position)^-1
    return r_mantle * r_paleomag.get_inverse()

records = []
for age in AGES:
    r_tpw = get_accumulated_tpw(age)
    if r_tpw.represents_identity_rotation():
        pole_lat, pole_lon, angle_deg = 90.0, 0.0, 0.0
    else:
        pole_ll, angle_rad = r_tpw.get_lat_lon_euler_pole_and_angle_radians()
        pole_lat, pole_lon = pole_ll
        angle_deg = float(np.degrees(angle_rad))
    records.append({"age_ma": float(age),
                    "pole_lat": float(pole_lat),
                    "pole_lon": float(pole_lon),
                    "angle_deg": float(angle_deg)})

tpw_df = pd.DataFrame(records)
print("First 6 accumulated TPW rotations:")
print(tpw_df.head(6).to_string(index=False))


AttributeError: 'FiniteRotation' object has no attribute 'get_lat_lon_euler_pole_and_angle_radians'

### How to read this table

- **age_ma** — the age at which we're evaluating the accumulated TPW correction.
- **pole_lat / pole_lon** — the Euler pole of the accumulated TPW rotation, in present-day geographic coordinates.
- **angle_deg** — the angle of that accumulated rotation. Row 0 is always 0° (no TPW at 0 Ma by definition).

The table grows monotonically only if TPW is monotonic — which it isn't. TPW is a physical process that can reverse, oscillate, and accelerate. Watch for non-monotonic behaviour.


## 3. Compute the incremental TPW rate

The accumulated rotation between adjacent time steps is the *incremental* TPW rotation over that 5-Myr interval:

**R_incr(t → t+Δt)** = R_TPW(t+Δt) × R_TPW(t)⁻¹

Its angle divided by Δt is the TPW rate in °/Myr — the physical quantity everyone quotes when they say "TPW peaked at 3 °/Myr in the mid-Cretaceous".


In [ ]:
incr_records = []
for i in range(1, len(AGES)):
    age_prev = AGES[i-1]
    age_curr = AGES[i]
    r_prev = get_accumulated_tpw(age_prev)
    r_curr = get_accumulated_tpw(age_curr)
    r_incr = r_curr * r_prev.get_inverse()
    if r_incr.represents_identity_rotation():
        angle_deg = 0.0
        pole_lat, pole_lon = 90.0, 0.0
    else:
        pole_ll, angle_rad = r_incr.get_lat_lon_euler_pole_and_angle_radians()
        pole_lat, pole_lon = pole_ll
        angle_deg = float(np.degrees(angle_rad))
    dt = age_curr - age_prev
    incr_records.append({
        "age_mid_ma": float((age_prev + age_curr) / 2),
        "dt_myr": float(dt),
        "angle_deg": float(angle_deg),
        "rate_deg_per_myr": float(angle_deg / dt),
        "pole_lat": float(pole_lat),
        "pole_lon": float(pole_lon),
    })
incr_df = pd.DataFrame(incr_records)
print(f"  Mean TPW rate: {incr_df['rate_deg_per_myr'].mean():.3f} °/Myr")
print(f"  Peak TPW rate: {incr_df['rate_deg_per_myr'].max():.3f} °/Myr "
      f"at ~{incr_df.loc[incr_df['rate_deg_per_myr'].idxmax(),'age_mid_ma']:.0f} Ma")


## 4. TPW pole path on an orthographic globe

Plot the sequence of accumulated TPW Euler poles as a track. Colour encodes age. A stationary track indicates monotonic TPW around a stable axis; a wandering track indicates the TPW rotation axis itself is drifting through time.


In [ ]:
# Filter to poles at ages where the rotation is meaningfully non-zero
plot_df = tpw_df[tpw_df["angle_deg"].abs() > 0.5].copy()

fig = pygmt.Figure()
fig.basemap(region=POLE_PATH_REGION, projection=POLE_PATH_PROJ, frame="ag")
fig.coast(region=POLE_PATH_REGION, projection=POLE_PATH_PROJ,
          land="gray90", water="white", shorelines="0.25p,gray50")

pygmt.makecpt(cmap="batlow", series=[0, MAX_AGE_MA, STEP_MA], reverse=True)
fig.plot(x=plot_df["pole_lon"], y=plot_df["pole_lat"],
         style="c0.28c", fill=plot_df["age_ma"], cmap=True, pen="0.4p,black",
         region=POLE_PATH_REGION, projection=POLE_PATH_PROJ)

# Track line connecting successive poles
fig.plot(x=plot_df["pole_lon"], y=plot_df["pole_lat"],
         pen="1.0p,gray40,-", region=POLE_PATH_REGION, projection=POLE_PATH_PROJ)

fig.colorbar(position="JBC+w10c/0.35c+h+o0/1c",
             frame=["a50f10", "x+lAge of accumulated TPW rotation (Ma)"])
fig.text(text=f"TPW pole path 0-{MAX_AGE_MA:.0f} Ma  ({MODEL_NAME})",
         position="TL", offset="0.25c/-0.25c", justify="TL",
         font="14p,Helvetica-Bold,black", fill="white", pen="0.6p,gray40")
fig.show(width=900)


### How to read the pole path

- Each dot is one accumulated-TPW Euler pole, colour-coded by age.
- **A cluster near a single point** means TPW has been rotating around the same axis for a sustained interval — the classic "TPW event" pattern.
- **A wandering track** means the rotation axis of the accumulated TPW correction is itself drifting through time.
- **Poles at high latitude** — TPW is rotating the Earth around a near-polar axis, i.e. relatively little apparent motion of continents in latitude.
- **Poles at low latitude** — TPW is rotating around a near-equatorial axis, driving continents through paleolatitude — the "true" polar wander case.

Steinberger & Torsvik (2008) find the Phanerozoic TPW pole clusters in the equatorial Indian Ocean / western Pacific — consistent with mass redistribution driven by African LLSVP + Pacific slab-graveyard mantle structure.


## 5. TPW rate through time — bar chart

Incremental TPW rate in °/Myr, one bar per 5-Myr interval. Highlights the well-documented ~110-100 Ma peak.

Steinberger & Torsvik (2008) hard-coded reference values (from their Fig 2 / Table 1) are overplotted for cross-checking. Your Z22-derived curve should broadly track theirs but may differ in detail because Z22 uses a different mantle-frame construction (Cao et al. 2024 mantle reference model) than Steinberger's paleomagnetic-Euler-pole approach.


In [ ]:
# Steinberger & Torsvik 2008 Fig 2 hand-digitised (age Ma, rate deg/Myr).
ST2008 = pd.DataFrame({
    "age_ma":           [10, 30, 60, 90, 110, 130, 150, 170, 200, 240, 280, 320],
    "rate_deg_per_myr": [0.5, 0.6, 0.9, 1.4, 1.9, 1.1, 0.8, 0.7, 0.6, 0.9, 1.3, 0.7],
})

fig = pygmt.Figure()
Y_MAX = max(float(incr_df["rate_deg_per_myr"].max()), float(ST2008["rate_deg_per_myr"].max())) + 0.3
fig.basemap(region=[MAX_AGE_MA, 0, 0, Y_MAX], projection="X20c/8c",
            frame=["WSne+t" + f"True polar wander rate through time — {MODEL_NAME} decomposition",
                   "xa50f10+lAge (Ma)",
                   "yaf+lTPW rate (° per Myr)"])

# Highlight ~110 Ma monster TPW event as a shaded rectangle
fig.plot(x=[100, 120, 120, 100, 100], y=[0, 0, Y_MAX, Y_MAX, 0],
         fill="gray@85", pen="0.3p,gray40")
fig.text(x=110, y=Y_MAX - 0.15, text="~110 Ma monster TPW event",
         font="8p,Helvetica-Bold,gray20", justify="MC", no_clip=True)

# Bars (Z22-derived rates)
fig.plot(x=incr_df["age_mid_ma"], y=incr_df["rate_deg_per_myr"],
         style=f"b{RATE_BAR_WIDTH_MA * 0.5}c",
         fill="#e67e22", pen="0.4p,black",
         label=f"Z22-derived TPW rate ({MODEL_NAME} mantle vs paleomag frame)")

# Steinberger reference curve
fig.plot(x=ST2008["age_ma"], y=ST2008["rate_deg_per_myr"],
         pen="2p,#2c3e50", label="Steinberger & Torsvik 2008 (Fig 2, hand-digitised)")
fig.plot(x=ST2008["age_ma"], y=ST2008["rate_deg_per_myr"],
         style="c0.28c", fill="#2c3e50", pen="0.3p,black")

fig.legend(position="JTR+jTR+o0.2c/0.2c", box="+gwhite+p0.5p,gray40")
fig.show(width=1100)


### How to read the rate plot

- **Bars** — Z22-derived TPW rate at each 5-Myr interval.
- **Dots + line** — Steinberger & Torsvik 2008 reference curve.
- **~110 Ma highlight** — the "monster" TPW event, driven by Ontong-Java Plateau eruption + Kerguelen LIP mass loading of the mantle. Rate peaks at ~2 °/Myr in the paleomag literature; Z22's decomposition should show a similar peak, though not necessarily identical amplitude because Z22's mantle-frame construction differs from Steinberger's.
- **Cretaceous quiescence (~90-70 Ma)** — TPW rate near zero. The classic "Cretaceous stillstand".
- **Late Paleozoic (~280-320 Ma)** — a secondary peak in Steinberger's curve, sometimes attributed to Pangaea assembly.

**Sanity check**: the mean Phanerozoic TPW rate in the literature is 0.5-1.0 °/Myr. If your curve is systematically an order of magnitude off, likely the anchor plate ID for the paleomag frame is wrong.

**Caveats**

- Z22's paleomag frame is defined via a synthetic 701701 anchor. This is Z22's own construction; it's not an independent paleomag reconstruction. Comparing Z22-derived TPW against Steinberger 2008 is comparing "Z22's internal TPW estimate" against "one specific published estimate", not an absolute measurement.
- Only using one plate (Africa) — TPW should be invariant across plates but numerical precision in the rotation-composition chain can produce ~10 % scatter.


## Extend this

- **Vary the plate.** Swap `PLATE_ID_MANTLE_FRAME` / `PLATE_ID_PALEOMAG_FRAME` to Australia (801), Antarctica (802), or Eurasia (301). The extracted TPW should be identical (up to numerical noise) because TPW is a global rotation. Any big disagreement is a sign of frame-chain issues.
- **Use Merdith 2021 instead of Z22.** Merdith 2021's default rotation file is in the paleomag frame with anchor 0; to get the mantle frame you'd need to compose with an APM reference frame (Torsvik & van der Voo 2013). More work but extends the analysis back to 1 Ga.
- **Compare against the Torsvik 2019 published TPW curve.** Digitise the values from Torsvik et al. (2019, *Earth & Planetary Science Letters*) or Torsvik & Cocks (2019, Cambridge book) and add as a third comparison series.
- **Plot TPW-corrected vs uncorrected continent positions.** For a chosen age, plot Africa in both frames as coloured polygons — the visual gap between them IS the accumulated TPW rotation.
- **Cross-reference with T61.** The TPW rate integrated over an interval predicts the paleoclimate shift T61 measures. High-TPW-rate intervals should show the largest T61 map-difference amplitudes.

## Related notebooks

- **T31** — paleo-mantle reference-frame uncertainty on continent positions. T63 is the mathematical origin of that uncertainty.
- **T61** — reference-frame uncertainty on reconstructed paleoclimate. T63 explains why T61 finds non-zero frame gaps.
- **T29-T30** — paleomagnetic reference-frame construction.

## Sources

- Steinberger, B. & Torsvik, T.H. (2008). Absolute plate motions and true polar wander in the absence of hotspot tracks. *Nature* 452, 620-623. doi:10.1038/nature06824.
- Torsvik, T.H., Steinberger, B., Ashwal, L.D., Doubrovine, P.V. & Trønnes, R.G. (2016). Earth evolution and dynamics — a tribute to Kevin Burke. *Canadian Journal of Earth Sciences* 53(11), 1073-1087.
- Torsvik, T.H. & Cocks, L.R.M. (2019). *Earth history and palaeogeography*. Cambridge University Press. Chapter 3 (True polar wander).
- Doubrovine, P.V., Steinberger, B. & Torsvik, T.H. (2012). Absolute plate motions in a reference frame defined by moving hot spots in the Pacific, Atlantic, and Indian oceans. *Journal of Geophysical Research: Solid Earth* 117, B09101. doi:10.1029/2011JB009072.
- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Fox, P. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9(2), 371-383. doi:10.1002/gdj3.146.
